# Đánh giá Offline (Render & Evaluate)

Notebook này gồm 2 bước:
1. **Inference (Render):** Chạy `render.py` để sinh ra ảnh từ point cloud (`.ply`).
2. **Evaluate:** Tính điểm (PSNR, SSIM, LPIPS) và Final Score theo chuẩn của BTC.

In [ ]:
# 1. Thay đổi tên scene và loại model (3dgs / 2dgs) ở đây
SCENE = 'HCM0204'
MODEL_TYPE = '3dgs'  # Đổi thành '2dgs' nếu bạn muốn chấm điểm 2DGS

MODEL_DIR = f'/kaggle/working/outputs_{MODEL_TYPE}/{SCENE}'
DATASET_PATH = f'/kaggle/input/datasets/ptquanh/vtar-b1-preprocessed-dataset/public_set/{SCENE}/train'

print(f"Tiến hành Render cho {MODEL_TYPE.upper()} - Scene {SCENE}...")

# Chuyển thư mục về repo tương ứng
import os
if MODEL_TYPE == '3dgs':
    %cd /kaggle/working/gaussian-splatting
else:
    %cd /kaggle/working/2d-gaussian-splatting

# Chạy lệnh render. Nó sẽ tự động đọc thư mục model_dir, lấy tập test_cameras (vì lúc train ta dùng cờ --eval)
# và lưu ảnh ra MODEL_DIR/test/renders/
!python render.py -m {MODEL_DIR} -s {DATASET_PATH}

In [ ]:
# 2. Cài đặt các thư viện cần thiết cho Render và Tính điểm
!pip install -q lpips scikit-image plyfile tqdm opencv-python joblib pillow

import os
import sys

try:
    import simple_knn
except ImportError:
    print("Đang cài đặt simple-knn...")
    !pip install -q submodules/simple-knn

if MODEL_TYPE == '3dgs':
    try:
        import diff_gaussian_rasterization
    except ImportError:
        print("Đang cài đặt diff-gaussian-rasterization...")
        !pip install -q submodules/diff-gaussian-rasterization
else:
    try:
        import diff_surfel_rasterization
    except ImportError:
        print("Đang cài đặt diff-surfel-rasterization...")
        !pip install -q submodules/diff-surfel-rasterization


In [ ]:
import cv2
import numpy as np
import torch
import lpips
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from tqdm.auto import tqdm

def load_image_for_skimage(path):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

def load_image_for_lpips(path):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = (img / 255.0) * 2.0 - 1.0
    tensor = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0)
    return tensor

In [ ]:
# ---------------------------------------------------------
# CẤU HÌNH ĐƯỜNG DẪN TÍNH ĐIỂM
# ---------------------------------------------------------
import glob

# Lệnh render.py sẽ tự động tạo thư mục theo số iteration, ví dụ: ours_15000 hoặc ours_30000
test_dir = f'{MODEL_DIR}/test'
ours_dirs = glob.glob(f'{test_dir}/ours_*')

if len(ours_dirs) > 0:
    # Lấy thư mục tìm thấy đầu tiên
    ours_dir = ours_dirs[0]
    GT_DIR = f'{ours_dir}/gt'
    RENDER_DIR = f'{ours_dir}/renders'
    print(f"Đã tìm thấy đường dẫn render tự động: {ours_dir}")
else:
    # Fallback dự phòng nếu không tìm thấy
    GT_DIR = f'{test_dir}/gt'
    RENDER_DIR = f'{test_dir}/renders'
    print("Cảnh báo: Không tìm thấy thư mục ours_*, dùng đường dẫn dự phòng.")

PSNR_MAX = 40.0

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
loss_fn_vgg = lpips.LPIPS(net='vgg').to(device)

psnr_list = []
ssim_list = []
lpips_list = []

if not os.path.exists(GT_DIR):
    print(f"Thư mục GT không tồn tại: {GT_DIR}")
elif not os.path.exists(RENDER_DIR):
    print(f"Thư mục Render không tồn tại: {RENDER_DIR}")
else:
    filenames = sorted([f for f in os.listdir(RENDER_DIR) if f.endswith(('.png', '.jpg'))])
    
    for fname in tqdm(filenames, desc="Evaluating"):
        gt_path = os.path.join(GT_DIR, fname)
        render_path = os.path.join(RENDER_DIR, fname)
        
        if not os.path.exists(gt_path):
            print(f"Không tìm thấy ảnh GT tương ứng cho {fname}")
            continue
            
        # Tính PSNR & SSIM
        gt_img = load_image_for_skimage(gt_path)
        render_img = load_image_for_skimage(render_path)
        
        if gt_img.shape != render_img.shape:
            render_img = cv2.resize(render_img, (gt_img.shape[1], gt_img.shape[0]))
            
        val_psnr = psnr(gt_img, render_img, data_range=255)
        val_ssim = ssim(gt_img, render_img, data_range=255, channel_axis=-1)
        
        # Tính LPIPS
        gt_tensor = load_image_for_lpips(gt_path).to(device)
        render_tensor = load_image_for_lpips(render_path).to(device)
        
        with torch.no_grad():
            val_lpips = loss_fn_vgg(gt_tensor, render_tensor).item()
            
        psnr_list.append(val_psnr)
        ssim_list.append(val_ssim)
        lpips_list.append(val_lpips)
        
    if len(psnr_list) > 0:
        avg_psnr = np.mean(psnr_list)
        avg_ssim = np.mean(ssim_list)
        avg_lpips = np.mean(lpips_list)
        
        psnr_norm = max(0.0, min(avg_psnr / PSNR_MAX, 1.0))
        score = 0.4 * (1 - avg_lpips) + 0.3 * avg_ssim + 0.3 * psnr_norm
        score_100 = score * 100
        
        print("\n" + "="*40)
        print(f"KẾT QUẢ ĐÁNH GIÁ {MODEL_TYPE.upper()} TRÊN {len(psnr_list)} ẢNH:")
        print("="*40)
        print(f"PSNR  : {avg_psnr:.4f} dB")
        print(f"SSIM  : {avg_ssim:.4f}")
        print(f"LPIPS : {avg_lpips:.4f}")
        print("-"*40)
        print(f"FINAL SCORE: {score_100:.2f} / 100")
        print("="*40)
    else:
        print("Không có ảnh hợp lệ nào được đánh giá!")

In [ ]:
# 3. Nén ảnh Render và GT để tải về kiểm tra (Tùy chọn)
import shutil
import os

zip_name = f'renders_{SCENE}'
zip_path = f'/kaggle/working/{zip_name}.zip'

# Tạo một thư mục tạm để copy cả 2 vào
temp_dir = f'/kaggle/working/{zip_name}'
os.makedirs(temp_dir, exist_ok=True)

if os.path.exists(GT_DIR):
    shutil.copytree(GT_DIR, f'{temp_dir}/gt', dirs_exist_ok=True)
if os.path.exists(RENDER_DIR):
    shutil.copytree(RENDER_DIR, f'{temp_dir}/renders', dirs_exist_ok=True)

print(f"Đang nén ảnh vào {zip_path}...")
shutil.make_archive(zip_path.replace('.zip', ''), 'zip', temp_dir)
shutil.rmtree(temp_dir)
print("Hoàn tất! Bạn có thể tải file zip về máy để xem.")